# Lab 3 — 에이전트 루프

> **이론 복습 — Session 3 슬라이드**
> - ReAct 루프: 추론(Reason) → 행동(Act) → 관찰(Observe) → 반복
> - 종료 조건: 모델이 도구를 더 안 부르면 멈춘다 / `MAX_STEPS` 안전망
> - 메모리 = `messages` 리스트. LLM 은 stateless, 기억은 우리가 운반한다.

## lab0 와의 연결

lab2 에서는 도구 한 바퀴(① 스키마 → ② 호출 결정 → ③ 실행 → ④ 결과) 만 돌렸습니다.
이번 lab 의 주제는 그 바퀴를 **여러 번** 돌리는 것입니다.

lab0 의 `src/lib/agent.js` ▶ `runAgent()` 가 정확히 이걸 합니다. 이번 lab 에서는 그 함수를
**Python 으로 재현** 합니다 — 줄 단위로 대조하면서 보세요.

| lab0 (JavaScript) | 이번 lab (Python) |
|---|---|
| `src/lib/tools.js` | 노트북에 그대로 인라인 |
| `src/lib/agent.js` ▶ `runAgent()` | `Agent.run()` (직접 만듦) |
| `MAX_STEPS = 6` | `max_steps = 6` |

## 학습 목표
1. lab2 의 "한 바퀴" 를 *반복하는* `Agent` 클래스를 완성한다
2. 도구가 **연쇄 호출** 되는 멀티스텝 질문을 직접 풀어 본다
3. lab0 챗봇에서 본 7번의 도구 호출이 실제로 *왜* 루프였는지 step 출력으로 확인한다


## 0. 준비

`labs/` 폴더에서 실행하세요. 키가 없으면 `MockLLM` 으로 동작합니다
(이 경우 에이전트는 도구를 한 번만 부르고 끝냅니다 — 메커니즘은 동일합니다).


In [ ]:
import json
from pathlib import Path
from common.llm import LLMClient

llm = LLMClient()

# lab0 데이터
DATA = Path("lab0_vibe_coding/data")
cities = json.loads((DATA / "cities.json").read_text(encoding="utf-8"))
flows  = json.loads((DATA / "flows.json").read_text(encoding="utf-8"))

CITY_BY_ID   = {c["id"]: c for c in cities}
CITY_BY_NAME = {c["name_ko"]: c["id"] for c in cities}

print(f"도시 {len(cities)}개, 흐름 {len(flows)}개")

## 1. 도구 5개 — lab0/lab2 와 동일

lab0 의 `src/lib/tools.js` 의 5개 함수를 Python 으로 옮긴 것입니다.
(읽어 보기만 하고 넘어가도 됩니다 — 이번 lab 의 주인공은 *루프* 입니다.)


In [ ]:
def _resolve(name_or_id):
    if not name_or_id: return None
    if name_or_id in CITY_BY_ID:   return name_or_id
    return CITY_BY_NAME.get(name_or_id)


def find_city(query: str) -> list[dict]:
    if not query: return []
    q = query.lower()
    return [
        {"id": c["id"], "name_ko": c["name_ko"], "name": c["name"],
         "lat": c["lat"], "lon": c["lon"]}
        for c in cities
        if q in c["name_ko"].lower() or q in c["name"].lower() or q == c["id"].lower()
    ]


def get_flow(origin: str, dest: str) -> dict:
    o, d = _resolve(origin), _resolve(dest)
    if not o or not d:
        return {"error": f"city not found: origin={origin}, dest={dest}"}
    for f in flows:
        if f["origin"] == o and f["dest"] == d:
            return {"origin": o, "dest": d, "count": f["count"]}
    return {"origin": o, "dest": d, "count": 0}


def get_round_trip(city_a: str, city_b: str) -> dict:
    a, b = _resolve(city_a), _resolve(city_b)
    if not a or not b:
        return {"error": f"city not found"}
    ab = next((f["count"] for f in flows if f["origin"]==a and f["dest"]==b), 0)
    ba = next((f["count"] for f in flows if f["origin"]==b and f["dest"]==a), 0)
    return {"city_a": CITY_BY_ID[a]["name_ko"], "city_b": CITY_BY_ID[b]["name_ko"],
            "a_to_b": ab, "b_to_a": ba, "total": ab + ba}


def get_city_totals(city: str) -> dict:
    cid = _resolve(city)
    if not cid: return {"error": f"city not found: {city}"}
    inflow  = sum(f["count"] for f in flows if f["dest"]   == cid)
    outflow = sum(f["count"] for f in flows if f["origin"] == cid)
    return {"city": cid, "name_ko": CITY_BY_ID[cid]["name_ko"],
            "inflow": inflow, "outflow": outflow, "total": inflow + outflow}


def get_top_flows(n: int = 5) -> list[dict]:
    return [
        {"origin": f["origin"], "dest": f["dest"], "count": f["count"],
         "label": f"{CITY_BY_ID[f['origin']]['name_ko']}→{CITY_BY_ID[f['dest']]['name_ko']}"}
        for f in sorted(flows, key=lambda x: -x["count"])[:n]
    ]


# 도구 스키마 — lab0/tools.js 의 TOOL_DECLARATIONS 와 동일.
SCHEMAS = [
    {"name": "find_city",
     "description": "Find cities whose Korean name, English name, or id contains the query. Returns id, name_ko, lat, lon.",
     "parameters": {"type": "object",
                    "properties": {"query": {"type": "string"}},
                    "required": ["query"]}},
    {"name": "get_flow",
     "description": "Directed commute volume from origin to dest. SEO→INC and INC→SEO are different items.",
     "parameters": {"type": "object",
                    "properties": {"origin": {"type": "string"}, "dest": {"type": "string"}},
                    "required": ["origin", "dest"]}},
    {"name": "get_round_trip",
     "description": "Round-trip commute between two cities (both directions summed).",
     "parameters": {"type": "object",
                    "properties": {"city_a": {"type": "string"}, "city_b": {"type": "string"}},
                    "required": ["city_a", "city_b"]}},
    {"name": "get_city_totals",
     "description": "Total commute inflow / outflow / sum (inflow+outflow) for one city.",
     "parameters": {"type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"]}},
    {"name": "get_top_flows",
     "description": "Return the N largest commute flows, largest first.",
     "parameters": {"type": "object",
                    "properties": {"n": {"type": "integer", "description": "default 5"}},
                    "required": []}},
]

TOOLBOX = {
    "find_city": find_city,
    "get_flow": get_flow,
    "get_round_trip": get_round_trip,
    "get_city_totals": get_city_totals,
    "get_top_flows": get_top_flows,
}

print(f"{len(SCHEMAS)}개 도구 준비")

## 2. 왜 루프가 필요한가 — 한 바퀴로는 안 풀리는 질문

lab2 의 `one_round()` 처럼 도구를 *한 번* 만 부르고 끝낸다고 생각해 봅시다.
그러면 다음 질문은 풀 수 없습니다:

> *"통근량이 가장 많은 구간을 찾고, 그 두 도시의 왕복 통근량도 알려줘."*

- 첫 번째 도구 `get_top_flows(1)` 결과를 *봐야* 어느 도시를 `get_round_trip` 에 넣을지 정해집니다.
- 즉, 두 번째 도구 호출은 첫 번째 결과에 **의존** 합니다.
- 한 바퀴로는 이 의존성을 풀 수 없습니다 — 그래서 **루프** 가 필요합니다.


## 3. 메모리 = `messages` 리스트

에이전트의 기억은 메시지 딕셔너리의 리스트입니다. 세 가지 역할(role)이 있습니다.

```
{"role": "user",      "content": "..."}                          ← 사용자 질문
{"role": "assistant", "content": "...", "tool_calls": [...]}     ← 모델의 응답
{"role": "tool",      "name": "...",   "content": "<결과 JSON>"}  ← 도구 실행 결과
```

매 스텝마다 모델은 이 리스트 **전체** 를 다시 보고 다음 행동을 정합니다.
LLM 은 stateless 다 — 기억은 우리가 운반한다.

lab0 의 `agent.js` 에서는 같은 역할을 Gemini 의 `contents` 배열이 합니다.
`role: 'user' | 'model'`, parts 에 `functionCall` / `functionResponse` — 모양만 다르고 개념은 같습니다.


## 4. `Agent` 클래스 — 🔧 이 셀에 TODO 가 있습니다

아래 `Agent.run()` 은 Session 3 슬라이드의 의사코드와 똑같습니다.
lab0/agent.js 의 `runAgent()` 와 줄 단위로 비교해 보세요.

한 곳, `_finalize()` 메서드만 비어 있습니다 — TODO 주석을 따라 완성하세요.


In [ ]:
class Agent:
    """LLM 에이전트 — 추론→행동→관찰 을 반복."""

    def __init__(self, llm, schemas, functions, system=None, max_steps=6):
        self.llm        = llm
        self.schemas    = schemas      # 모델이 보는 도구 스키마
        self.functions  = functions    # {이름: 실제 함수}
        self.system     = system
        self.max_steps  = max_steps

    def _run_tool(self, name, args):
        """도구 하나 실행. 에러는 *던지지 않고* 결과로 돌려준다 — 모델이 보고 복구할 수 있게."""
        if name not in self.functions:
            return {"error": f"unknown tool: {name}"}
        try:
            return self.functions[name](**(args or {}))
        except Exception as exc:
            return {"error": f"{type(exc).__name__}: {exc}"}

    def _finalize(self, messages):
        # 🔧 TODO: 루프가 max_steps 를 다 써도 안 끝났을 때 호출됨.
        # 포기하지 말고, 도구 없이 LLM 을 한 번 더 호출해 가진 정보로 최선의 답을 받는다.
        # 힌트: self.llm.generate(messages, self.system) 이 LLMResponse 를 반환.
        return "(최대 스텝에 도달했습니다.)"   # <-- TODO: 이 줄을 바꾸세요

    def run(self, question, verbose=True):
        """질문을 받아 추론→행동→관찰을 반복해 답을 만든다."""
        messages = [{"role": "user", "content": question}]

        for step in range(self.max_steps):
            # ── 추론(Reason): 모델이 답할지 / 도구를 부를지 결정
            reply = self.llm.generate_with_tools(messages, self.schemas, self.system)

            # 도구를 더 안 부르면 → 끝
            if not reply.wants_tool:
                if verbose:
                    print(f"  step {step + 1}: 텍스트 답변으로 종료")
                return reply.text

            messages.append({"role": "assistant", "content": reply.text,
                             "tool_calls": reply.tool_calls})

            # ── 행동(Act) + 관찰(Observe): 도구 실행하고 결과를 기록
            for call in reply.tool_calls:
                result = self._run_tool(call.name, call.args)
                if verbose:
                    print(f"  step {step + 1}: {call.name}({call.args})")
                messages.append({
                    "role": "tool", "name": call.name,
                    "content": json.dumps(result, ensure_ascii=False, default=str),
                })

        # 안전망: max_steps 에 닿음
        return self._finalize(messages)

print("Agent 클래스 정의 완료.")

## 5. 단일 스텝 질문 — 비교 기준

먼저 도구 한 번이면 풀리는 질문입니다. `step 1: …` 출력으로 **루프가 1번만 돌고 끝나는** 것을 관찰하세요.


In [ ]:
SYSTEM = (
    "You are a 수도권 통근 데이터 안내 도우미. "
    "Always answer in Korean, concise (2~4 sentences). "
    "Use tools — never guess numbers."
)

agent = Agent(llm, schemas=SCHEMAS, functions=TOOLBOX, system=SYSTEM, max_steps=6)

print("--- 단일 스텝 질문 ---")
answer = agent.run("서울에서 인천으로 가는 통근량은?")
print()
print("답변:", answer)

## 6. 멀티스텝 질문 ① — BUILD_SPEC §4-4 마지막 줄

이제 도구가 **두 번 이상** 필요한 질문입니다. lab0 의 검증 대화 마지막 줄이기도 합니다.
실제 키로 돌리면 `step 1: get_top_flows(...)` → `step 2: get_round_trip(...)` 식으로
*도구가 연쇄* 되는 것이 보입니다.


In [ ]:
print("--- 멀티스텝 ① ---")
answer = agent.run(
    "통근량이 가장 많은 구간을 찾고, 그 두 도시 사이 왕복 통근량도 알려줘."
)
print()
print("답변:", answer)

## 7. 멀티스텝 질문 ② — lab0 챗봇에서 실제로 본 그 질문

방금 lab0 챗봇에서 다음 질문을 던졌더니, 도구 호출이 **7번** 일어났습니다:

> *"통근량이 가장 많은 상위 3개 알려주고, 이 도시들 사이의 통근량 알려줘."*

그때 모델은 "루프를 안 돌았다" 고 답했는데 — *그건 모델의 환각* 입니다.
루프 없이는 가능하지 않은 동작이에요. 직접 돌려 보고 step 출력으로 확인하세요.


In [ ]:
print("--- 멀티스텝 ② (사용자 실제 질문) ---")
answer = agent.run(
    "통근량이 가장 많은 상위 3개 구간 알려주고, "
    "그 구간에 등장한 도시들 사이의 단방향 통근량을 모두 알려줘."
)
print()
print("답변:", answer)

**관찰 포인트**

- step 1: 모델은 *어느* 도시들이 상위인지 모르므로, 먼저 `get_top_flows(n=3)` 을 부른다.
- step 2: 결과(서울/성남/인천)를 *본 뒤* 비로소 `get_flow` 를 여러 번 부른다.
  - Gemini 는 한 응답에서 여러 functionCall 을 *동시에* 낼 수 있으므로,
    `get_flow` 6개가 step 2 한 줄 안에 묶여 나올 수 있습니다.
- step 3: 결과들을 받아 최종 텍스트로 답한다.

→ **step 1 의 결과가 step 2 의 입력을 결정한다.** 이게 한 바퀴(`one_round()`)로는 풀 수 없는 이유입니다.


## 8. lab0 의 `runAgent()` 와 줄별 대조

lab0/example_solution/src/lib/agent.js 를 같이 열어 두고 비교해 보세요.

| 개념 | 이 노트북 `Agent` | lab0 `runAgent` |
|---|---|---|
| 메모리 | `messages` (list of dict) | `contents` (Gemini contents 배열) |
| 한 바퀴 LLM 호출 | `self.llm.generate_with_tools(...)` | `callGemini({contents, tools})` |
| 종료 조건 | `not reply.wants_tool` | `calls.length === 0` |
| 안전망 | `_finalize()` (TODO) | `tools: []` 로 한 번 더 호출 |
| 도구 실행 | `_run_tool(name, args)` | `runTool(name, args)` |
| 결과 기록 | `{role: 'tool', name, content}` | `{role: 'user', parts: [{functionResponse}]}` |

언어/모양만 다르고 **로직은 동일** 합니다. 진짜로 한 번 비교해 보세요.


## 9. 🔧 추가 연습 — 더 어려운 멀티스텝

아래 질문 중 하나를 골라 `agent.run()` 으로 돌려 보세요. step 출력으로 도구가
어떻게 *순서대로* 불리는지 관찰하세요.


In [ ]:
harder_questions = [
    # find_city 로 좌표 회수 → get_round_trip
    "용인의 좌표를 알려주고, 용인과 수원 사이 왕복 통근량도 같이 알려줘.",

    # get_top_flows → 각 도시의 get_city_totals 비교
    "상위 2개 구간에 등장하는 도시들의 총 통근량(inflow+outflow)을 비교해줘. 어느 도시가 더 바쁘지?",

    # 데이터에 없는 도시 — 모델이 "모른다" 고 솔직히 답하는지 확인
    "강남과 송파 사이 통근량을 알려줘.",
]

for q in harder_questions:
    print("="*70)
    print("Q:", q)
    print("-"*70)
    print(agent.run(q))
    print()

## 정리

- `Agent` = lab2 의 한 바퀴를 `for step in range(MAX_STEPS)` 로 **반복**
- 종료 조건(`not wants_tool`) + 안전망(`_finalize`) 으로 루프는 *항상* 끝난다
- 멀티스텝 질문은 **앞 단계 결과가 다음 단계 입력을 결정** 하기 때문에 루프가 필요하다
- lab0 의 챗봇에서 본 "도구 호출 7회" 도 정확히 이 루프가 돌아서 나온 결과다

> 이 `Agent` 클래스의 완성본은 `common/agent.py` 에 있습니다 —
> 다음 실습(lab 4, 5)은 거기서 `from common.agent import Agent` 로 가져다 씁니다.

**다음 — Session 4**: 에이전트를 여러 개로 나누고(멀티에이전트), 지식을 주고(RAG),
표준화하고(MCP), 테스트하는 법.
